In [ ]:
# IMPORT PACKAGES
import os
import tempfile
import anndata # data container for single-cell omics, a flexible format storing: cell-by-gene expression matrices, metadata, embeddings (PCA, UMAP)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc # single-cell data analysis, built on top of anndata, clean and explore data
import scrublet as scr # Single-cell RNA-seq unsupervised Doublet Detection
import scvi # Deep probabilistic    Modeling for Single-Cell data, Apply scvi to model the data and uncover hidden patterns
import seaborn as sns # statistical data visualization, python library built on matplotlib
import torch # Deep learning framework for building and training neural networks

import igraph
import leidenalg
from scvi.model import SCVI, SCANVI, TOTALVI


import json
import pickle
import numpy as np
from pathlib import Path
from IPython.display import display, Image, HTML
import matplotlib.image as mpimg


import scipy.sparse as sp
import polars as pl
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import glob
from sklearn.metrics import balanced_accuracy_score
from typing import List
import gc

scvi.settings.seed = 0


In [ ]:
# query data should be either: gsa, lsa, pnas

#########################  
# PIPELINE OF SCVI/SCANVI
######################### 

class ML_Pipeline:  
    # constructor that initializes the pipeline with user parameters 
    def __init__(self, 
            ref_input = None, 
            latent_list: List[int] = None, 
            n_neighbors_list: List[int] = None, 
            make_gdt_adata: bool = False # make processed gdT data with "count" layer?
            ): 
        
        if ref_input is None: 
            self.ref_input = ["gsa", "lsa"]
        elif not isinstance(ref_input, list):
            raise ValueError(f"ref_input must be a list of dataset names, got {type(ref_input)}")
        else:
            self.ref_input = sorted(ref_input)

        self.make_gdt_adata = make_gdt_adata
        
        if latent_list is None: 
            self.latent_list = [10] # the default value 
        else: 
            self.latent_list = latent_list
        
        if n_neighbors_list is None: 
            self.n_neighbors_list = [30] # the default value 
        else: 
            self.n_neighbors_list = n_neighbors_list

        self.parent_dir = os.getcwd() 

        self.raw_gdt_adata_setup()


    # runs the processed data. If the make_gdt_adata is false and there is no processed data, it makes processed data from the raw data (adds the "count" layer)
    def raw_gdt_adata_setup(self): 
        sc.set_figure_params(figsize=(6, 6), frameon=False)
        sns.set_theme()

        processed_path = os.path.join(self.parent_dir, "processedData/processed_gdt_adata.h5ad")
        
        # if the processed path exist, load it 
        if self.make_gdt_adata==False and os.path.exists(processed_path): 
            self.gdt_adata_path = processed_path
        else: 
            raw_path = os.path.join(self.parent_dir, "rawData/gdtcells_rawData.h5ad")
            adata = sc.read(raw_path)

            if adata.raw is None:
                adata.layers["counts"] = adata.X.copy()
            else:
                adata.layers["counts"] = adata.raw.X.copy()

            if sp.issparse(adata.layers["counts"]):
                adata.layers["counts"].data = np.round(adata.layers["counts"].data)
            else:
                adata.layers["counts"] = np.round(adata.layers["counts"])
            
            # creates and save the processed file 
            os.makedirs(os.path.dirname(processed_path), exist_ok=True)
            adata.write(processed_path)
            self.gdt_adata_path = processed_path

    # Runs multiple query experiements in sequence 
    def run_pipeline(self, queryList):

        self.ref_str = "_".join(self.ref_input) # e.g., "lsa_gsa"

        for query_str in queryList: 

            # set the save directory and make it if it isn't there 
            self.save_dir = os.path.join(self.parent_dir, f"{self.ref_str}_ref_{query_str}_query")
            os.makedirs(self.save_dir, exist_ok=True)

            # set query_input as the user's input
            self.query_input = [query_str]

            # store all figures in e.g., dir/lsa_gsa_ref_pnas_query/figures/
            sc.settings.figdir = os.path.join(self.save_dir, "figures")
            os.makedirs(sc.settings.figdir, exist_ok=True)

            # run the pipeline 
            self.scvi_scanvi_ml(query_str)

            # Force clear after every query string
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    # executes the complete scVI -> scANVI analysis pipeline 
    def scvi_scanvi_ml(self, query_str): 

        query_group = f"{query_str}pbmc"

        # Create ref groups from ref_input list  
        ref_groups = [f"{r}pbmc" for r in self.ref_input]

        gdt_adata = sc.read(self.gdt_adata_path)

        # Create masks 
        groups = gdt_adata.obs["group"].astype(str)
        query_mask = (groups == query_group).to_numpy()
        ref_mask = groups.isin(ref_groups).to_numpy()
        
        # Remove any overlap 
        overlap = query_mask & ref_mask
        if overlap.sum() > 0:
            raise ValueError(
                f"Found {overlap.sum()} overlapping cells between query and reference. "
                f"This will cause data leakage. Check your group assignments."
            )
        
        ref_data = gdt_adata[ref_mask].copy()
        query_data = gdt_adata[query_mask].copy()

        sc.pp.highly_variable_genes(
            ref_data, 
            n_top_genes=2000, 
            batch_key="group", 
            subset=True, 
            layer = "counts",
            flavor="seurat_v3"
        )

        query_data = query_data[:, ref_data.var_names].copy()

        SCVI.setup_anndata(ref_data, batch_key="group", layer="counts")

        for latent in self.latent_list: 
            
            ## SCVI ML 

            # reference umap 
            scvi_ref, scvi_ref_path = self.load_or_train_scvi_ref(ref_data, latent)
            self.UMAP_visualization(ref_data, scvi_ref, latent, whichData = "ref", isSCVI = True)
            
            # query umap
            scvi_query = self.load_or_train_scvi_query(query_data, latent, scvi_ref_path)
            self.UMAP_visualization(query_data, scvi_query, latent, whichData = "query", isSCVI = True)

            # full umap
            scvi_full_data = anndata.concat([query_data, ref_data], keys=["query","ref"])
            self.UMAP_visualization(scvi_full_data, scvi_query, latent, whichData = "full", isSCVI = True)
            del scvi_full_data
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            ## SCANVI ML

            # reference umap
            scanvi_ref, scanvi_ref_path = self.load_or_train_scanvi_ref(ref_data, scvi_ref, latent)
            self.UMAP_visualization(ref_data, scanvi_ref, latent, whichData = "ref", isSCVI = False)

            # query umap
            scanvi_query, predicted_query_data = self.load_or_train_scanvi_query(query_data, latent, scanvi_ref_path)
            self.conf_mat_heatmap(predicted_query_data, latent, whichData = "query")
            
            # full umap
            scanvi_full_data = self.scanvi_full(predicted_query_data, ref_data, scanvi_query, latent)
            self.UMAP_visualization(scanvi_full_data, scanvi_query, latent, whichData = "full", isSCVI = False)

            # overlay umap with the sensitivity and specificity
            self.overlay_UMAP_visualization(scanvi_full_data, latent, query_str)

            del scanvi_full_data
            del predicted_query_data
            del scvi_ref, scvi_query, scanvi_ref, scanvi_query
            
            plt.close('all')

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        del ref_data, query_data
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Trains or loads pre-trained scVI model on reference data 
    def load_or_train_scvi_ref(self, ref_data, latent): 

        scvi_ref_path = os.path.join(self.save_dir, "scvi", "ref", f"latent{latent}")

        if os.path.exists(scvi_ref_path):
            scvi_ref = SCVI.load(scvi_ref_path, adata=ref_data)
        else: 
            scvi_ref = SCVI(
                ref_data,
                n_latent = latent,
                use_layer_norm="both",
                use_batch_norm="none",
                encode_covariates=True,
                dropout_rate=0.2,
                n_layers=2,
            )
            scvi_ref.train()
            os.makedirs(os.path.dirname(scvi_ref_path), exist_ok=True)
            scvi_ref.save(scvi_ref_path, overwrite = True)
        return scvi_ref, scvi_ref_path
    

    # Maps query data to pre-trained scVI model 
    def load_or_train_scvi_query(self, query_data, latent, scvi_ref_path): 
        
        SCVI_LATENT_KEY = "X_scVI"

        scvi_query_path = os.path.join(self.save_dir, "scvi", "query", f"latent{latent}")

        if os.path.exists(scvi_query_path):
            scvi_query = SCVI.load(scvi_query_path, adata=query_data)
        else: 
            # used to clear old scVI state
            if "_scvi" in query_data.uns:
                del query_data.uns["_scvi"]

            SCVI.prepare_query_anndata(query_data, scvi_ref_path)

            scvi_query = SCVI.load_query_data(
                query_data,
                scvi_ref_path,
            )

            scvi_query.train(max_epochs=200, plan_kwargs={"weight_decay": 0.0})
            os.makedirs(os.path.dirname(scvi_query_path), exist_ok=True)
            scvi_query.save(scvi_query_path, overwrite = True)

        query_data.obsm[SCVI_LATENT_KEY] = scvi_query.get_latent_representation()
        return scvi_query


    # Trains SCANVI classifier on labeled reference data
    def load_or_train_scanvi_ref(self, ref_data, scvi_ref, latent): 
        
        SCANVI_LABELS_KEY = "labels_scanvi"

        ref_data.obs[SCANVI_LABELS_KEY] = ref_data.obs["class"].astype(str).values
        
        scanvi_ref_path = os.path.join(self.save_dir, "scanvi", "ref", f"latent{latent}")
        
        if os.path.exists(scanvi_ref_path):
            scanvi_ref = SCANVI.load(scanvi_ref_path, adata=ref_data)
        else: 
            scanvi_ref = SCANVI.from_scvi_model(
            scvi_ref, # Trained SCVI model 
            unlabeled_category="Unknown", 
            labels_key=SCANVI_LABELS_KEY, 
            )

            scanvi_ref.train(max_epochs=20, n_samples_per_label=100)
            os.makedirs(os.path.dirname(scanvi_ref_path), exist_ok=True)
            scanvi_ref.save(scanvi_ref_path, overwrite = True)
        
        return scanvi_ref, scanvi_ref_path


    # Predicts cell types for query data using SCANVI     
    def load_or_train_scanvi_query(self, query_data, latent, scanvi_ref_path): 

        if "_scvi" in query_data.uns:
            del query_data.uns["_scvi"]

        SCANVI.prepare_query_anndata(query_data, scanvi_ref_path)
        scanvi_query = SCANVI.load_query_data(query_data, scanvi_ref_path)

        scanvi_query_path = os.path.join(self.save_dir, "scanvi", "query", f"latent{latent}")

        if os.path.exists(scanvi_query_path):
            scanvi_query = SCANVI.load(scanvi_query_path, adata=query_data)
        else: 
            scanvi_query.train(
                max_epochs=100,
                plan_kwargs={"weight_decay": 0.0},
                check_val_every_n_epoch=10,
            )

            os.makedirs(os.path.dirname(scanvi_query_path), exist_ok=True)
            scanvi_query.save(scanvi_query_path, overwrite = True)

        SCANVI_PREDICTIONS_KEY = "predictions_scanvi"

        query_data.obsm["X_scANVI"] = scanvi_query.get_latent_representation()
        query_data.obs[SCANVI_PREDICTIONS_KEY] = scanvi_query.predict()

        return scanvi_query, query_data 
    

    # Evaluates SCANVI predictions on full dataset
    def scanvi_full(self, predicted_query_data, ref_data, scanvi_query, latent): 
        
        # path made but not used - written if wish to store the accuracy results later 
        scanvi_full_path = os.path.join(self.save_dir, "scanvi", "full", f"latent{latent}")
        
        full = anndata.concat([predicted_query_data, ref_data], keys=["Query","Reference"], label="batch")
        full_predictions = scanvi_query.predict(full)
        full.obs["predictions_scanvi"] = full_predictions
        full_acc = np.mean(full_predictions == full.obs['class'])

        return full


    # generates standard UMAP visualization
    def UMAP_visualization(self, data, model, latent, whichData, isSCVI): 
        
        # set LATENT_KEY
        if isSCVI: 
            LATENT_KEY = "X_scVI"
            whichModel = "scvi"
        else: 
            LATENT_KEY = "X_scANVI"
            whichModel = "scanvi"

        data.obsm[LATENT_KEY] = model.get_latent_representation(data)

        for nn in self.n_neighbors_list:
            data_copy = data.copy()
            sc.pp.neighbors(data_copy, use_rep=LATENT_KEY, n_neighbors = nn)
            sc.tl.leiden(data_copy)
            sc.tl.umap(data_copy) # creates "X_umap" automatically 

            # save the coordinates
            umap_key = f"X_umap_{whichModel}_{whichData}_nn{nn}_latent{latent}"
            data.obsm[umap_key] = data_copy.obsm["X_umap"].copy()

            # -----Make normal UMAP-----
            umap_fig = sc.pl.umap(
                data_copy,
                color=["class", "group"],
                palette = "tab20",
                frameon=False,
                ncols=1,
                show = False, 
                return_fig = True
            )
            
            # save the figure
            run_key = f"umap_nn{nn}_latent{latent}"

            umap_fig_path = self.save_fig(umap_fig, whichData, isSCVI, "umap", run_key)
            plt.close(umap_fig)
        

            # -----Make colored UMAP----- 
            colored_umap_fig = sc.pl.umap(
                data_copy,
                color=["class"],
                palette = {
                        "gdT_1": "#d62728", 
                        "gdT_2": "#FFDA03",
                        "gdT_3": "#2ca02c", 
                        "gdT_4": "#1f77b4", 
                        "gdT_5": "#9467bd",
                        "gdT": "#ED7014", 
                        "0": "#876E4B", 
                        0: "#876E4B"
                }, 
                na_color = "#C0C0C0",
                frameon=False,
                ncols=1,
                show = False, 
                return_fig = True
            )

            run_key = f"colored_umap_nn{nn}_latent{latent}"

            colored_umap_fig_path = self.save_fig(colored_umap_fig, whichData, isSCVI, "colored_umap", run_key)
            plt.close(colored_umap_fig)


# Creates overlay UMAP showing query gdT cells on full dataset 
    def overlay_UMAP_visualization(self, full_data, latent, query_str, whichModel = "scanvi", whichData = "full"):
        
        for nn in self.n_neighbors_list:
            
            # Use to call the standard umap coordinates 
            umap_key = f"X_umap_{whichModel}_{whichData}_nn{nn}_latent{latent}"

            # Used to name the figure in the file 
            run_key = f"overlay_umap_nn{nn}_latent{latent}"

            if umap_key not in full_data.obsm:
                raise ValueError(f"Missing {umap_key} in full_data.obsm")

            if "batch" not in full_data.obs:
                raise ValueError("Expected `batch` in full_data.obs (Query/Reference).")
            
            # filters out whether each cell is from query or not 
            is_query_list = full_data.obs["batch"].astype(str).str.lower().isin(["query"])

            query_data = full_data[is_query_list].copy()
            
            # binary labels 
            is_gdt_true = query_data.obs["class"].astype(str).str.startswith("gdT")
            is_gdt_pred = query_data.obs["predictions_scanvi"].astype(str).str.startswith("gdT")
        
            # metrics 
            TP = ((is_gdt_true) & (is_gdt_pred)).sum()
            FN = ((is_gdt_true) & (~is_gdt_pred)).sum()
            TN = ((~is_gdt_true) & (~is_gdt_pred)).sum()
            FP = ((~is_gdt_true) & (is_gdt_pred)).sum()
            print(f"TP: {TP}, FN: {FN}, TN: {TN}, FP: {FP}")

            sensitivity = TP / (TP + FN) if (TP + FN) > 0 else np.nan
            specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

            # sets the size to draw the plot
            overlay_umap_fig, ax = plt.subplots(figsize=(11, 8))
            overlay_umap_fig.set_facecolor("#f6f6f6")
            ax.set_facecolor("#f6f6f6")

            # background: reference cells
            ax.scatter(
                full_data.obsm[umap_key][~is_query_list, 0],
                full_data.obsm[umap_key][~is_query_list, 1],
                c="#8f8f8f",
                s=6,
                alpha=0.9,
                label="Reference cells",
            )

            tp_mask = is_gdt_true & is_gdt_pred     # True Positives (gdT correctly predicted)
            fn_mask = is_gdt_true & ~is_gdt_pred    # False Negatives (gdT missed)

            # incorrect first
            if fn_mask.sum() > 0:
                ax.scatter(
                        query_data.obsm[umap_key][fn_mask, 0],
                        query_data.obsm[umap_key][fn_mask, 1],
                        c="#ea5a4f",
                        s=6,
                        alpha=0.9,
                        label="Query gdT missed (FN)",
                )

            # correct on top
            if tp_mask.sum() > 0:
                ax.scatter(
                        query_data.obsm[umap_key][tp_mask, 0],
                        query_data.obsm[umap_key][tp_mask, 1],
                        c="#659962",
                        s=6,
                        alpha=0.9,
                        label="Query gdT correctly predicted (TP)",
                )

            # labels 
            ax.set_title(
                f"{self.ref_str.upper()} → {query_str.upper()} - gdT Cell Prediction Accuracy",
                fontsize=16,
                fontweight="bold",
            )
            ax.set_xlabel("UMAP1")
            ax.set_ylabel("UMAP2")
            ax.legend(frameon=True)

            text = (
                f"gdT Sensitivity: {sensitivity*100:.1f}%\n"
                f"({TP}/{TP+FN})\n\n"
                f"gdT Specificity: {specificity*100:.1f}%\n"
                f"({TN}/{TN+FP})"
            )

            ax.text(
                0.02,
                0.98,
                text,
                transform=ax.transAxes,
                va="top",
                fontsize=12,
                bbox=dict(boxstyle="round", facecolor="#f6f6f6", alpha=0.85),
            )

            plt.tight_layout()

            overlay_umap_fig_path = self.save_fig(overlay_umap_fig, whichData, isSCVI=False, whichFig="overlay_umap", key=run_key)
            plt.close(overlay_umap_fig)


    # Saves UMAP results in organized format
    def save_fig(self, fig, whichData, isSCVI, whichFig, key):

        # set model name 
        if isSCVI: 
            whichModel = "scvi"
        else: 
            whichModel = "scanvi"
        
        # set path to save UMAP
        # whichFig: "umap", "colored_umap", "overlay_umap", "heatmap"
        # whichModel: "scvi", "scanvi"
        # whichData: "ref", "query", "full"
        fig_dir = os.path.join(sc.settings.figdir, whichFig, whichModel, whichData)
                
        os.makedirs(fig_dir, exist_ok=True)

        # key: e.g., "UMAP_nn30_latent10"
        fig_path = os.path.join(fig_dir, f"{key}.png")
        
        fig.savefig(fig_path, dpi=200, bbox_inches="tight")

        return fig_path


    # Creates confusion matrix heatmap of predictions
    def conf_mat_heatmap(self, predicted_query_data, latent, whichData): 

        SCANVI_PREDICTIONS_KEY = "predictions_scanvi"

        for nn in self.n_neighbors_list:
            df = predicted_query_data.obs.groupby(["class", SCANVI_PREDICTIONS_KEY]).size().unstack(fill_value=0)
            norm_df = df.div(df.sum(axis=0).replace(0, np.nan), axis=1).fillna(0)

            # Make the figure 
            plt.figure(figsize=(8, 8))
            _ = plt.pcolor(norm_df)
            _ = plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
            _ = plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
            plt.xlabel("Predicted")
            plt.ylabel("Observed")

            run_key = f"heatmap_nn{nn}_latent{latent}"

            heatmap_fig = plt.gcf() 
            heatmap_fig_path = self.save_fig(heatmap_fig, whichData, isSCVI=False, whichFig="heatmap", key=run_key)
            plt.close(heatmap_fig)

In [ ]:
pipeline_l_gp = ML_Pipeline(
    ref_input=["lsa"],
    latent_list=[10],
    n_neighbors_list=[30],
    make_gdt_adata=False
)
pipeline_l_gp.run_pipeline(["gsa", "pnas"])

In [ ]:
pipeline_l = ML_Pipeline(
    ref_input=["lsa"],
    latent_list=[10],
    n_neighbors_list=[30],
    make_gdt_adata=False
)
pipeline_l.run_pipeline(["gsa", "pnas"])


pipeline_gl_p = ML_Pipeline(
    ref_input=["lsa", "gsa"],
    latent_list=[10],
    n_neighbors_list=[30],
    make_gdt_adata=False
)
pipeline_gl_p.run_pipeline(["pnas"])


pipeline_lp_g = ML_Pipeline(
    ref_input=["lsa", "pnas"],
    latent_list=[10],
    n_neighbors_list=[30],
    make_gdt_adata=False
)
pipeline_lp_g.run_pipeline(["gsa"])


pipeline_gp_l = ML_Pipeline(
    ref_input=["gsa", "pnas"],
    latent_list=[10],
    n_neighbors_list=[30],
    make_gdt_adata=False
)
pipeline_gp_l.run_pipeline(["lsa"])

In [ ]:
def display_fig(
    ref_name: str, # "lsa", "gsa", "lsa_gsa", etc.
    query_name: str, # "pnas", "gsa", "lsa"
    whichFig: str,    # "umap", "colored_umap", "overlay_umap", "heatmap"
    whichModel: str,  # "scvi" or "scanvi"
    whichData: str,   # "ref", "query", "full"
    nn: int = 30,
    latent: int = 10,
    base_dir: str = ".",
):

    assert whichFig in {"umap", "colored_umap", "overlay_umap", "heatmap"}, \
        f"Invalid whichFig: {whichFig}"
    assert whichModel in {"scvi", "scanvi"}, \
        f"Invalid whichModel: {whichModel}"
    assert whichData in {"ref", "query", "full"}, \
        f"Invalid whichData: {whichData}"
    assert query_name in {"pnas", "gsa", "lsa"}, \
        f"Invalid query_name: {query_name}"
    
    ref_list = ref_name.split("_")
    ref_list = sorted(ref_list)
    ref_name = "_".join(ref_list)

    # Build experiment directory
    exp_dir = Path(base_dir) / f"{ref_name}_ref_{query_name}_query" / "figures"

    # Build figure directory
    fig_dir = exp_dir / whichFig / whichModel / whichData

    # Build filename
    fig_name = f"{whichFig}_nn{nn}_latent{latent}.png"

    fig_path = fig_dir / fig_name

    if not fig_path.exists():
        raise FileNotFoundError(f"Figure not found:\n{fig_path}")

    print(f"Displaying: {fig_path}")
    display(Image(filename=str(fig_path)))


In [ ]:
display_fig("lsa", "gsa", "overlay_umap", "scanvi", "full")